In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data = pd.read_csv('dataset.csv')

In [ ]:
BASE_PRICE = 10.0
LAMBDA = 0.5

ALPHA = 1.5
BETA = 1.0
GAMMA = 1.2
DELTA = 2.0
EPSILON = {
    'car': 1.0,
    'bike': 0.5,
    'truck': 1.5
}

if 'price' not in data.columns:
    data['price'] = BASE_PRICE

from sklearn.preprocessing import MinMaxScaler

def normalize_series(series):
    scaler = MinMaxScaler()
    return scaler.fit_transform(series.values.reshape(-1, 1)).flatten()

def compute_demand(row):
    occ_rate = row['occupancy'] / row['capacity'] if row['capacity'] > 0 else 0
    vehicle_weight = EPSILON.get(row['vehicle_type'], 1.0)
    demand = (
        ALPHA * occ_rate +
        BETA * row['queue_length'] -
        GAMMA * row['traffic_level'] +
        DELTA * row['is_special_day'] +
        EPSILON.get(row['vehicle_type'], 1.0)
    )
    return demand

data['raw_demand'] = data.apply(compute_demand, axis=1)
data['normalized_demand'] = normalize_series(data['raw_demand'])

def compute_price(demand):
    multiplier = 1 + LAMBDA * demand
    multiplier = min(max(multiplier, 0.5), 2.0)
    return round(BASE_PRICE * multiplier, 2)

data['price'] = data['normalized_demand'].apply(compute_price)

output_filename = 'demand_based_model_output.csv'
data.to_csv(output_filename, index=False)
print(f"Output saved to {output_filename}")
